# P-CULTA — 17-Example Utterance-Only Few-Shot Evaluation
## 4 Models | CAS · FTV (independent) · Bias

**Prompt condition:** Each of the 17 demonstrations contains only the user utterance, context and its gold response. The target contains only the user utteranceand context. No topic, roles, power distance, register, or sensitivity is shown to the model.

## Experimental control

This notebook preserves the generation and evaluation settings of the utterance_context baseline. The only change is the addition of a fixed pool of 17 utterance-only demonstrations.

## 0. Install & Import

In [ ]:
# import sys

# !{sys.executable} -m pip install -U \
# transformers \
# accelerate \
# bitsandbytes \
# sentencepiece \
# huggingface_hub

In [1]:
# !pip install openai transformers accelerate bitsandbytes pandas scipy scikit-learn matplotlib seaborn tqdm -q
# !pip install -q openai
import os, time, warnings, json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from collections import Counter

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--'
})

COLORS = {
    'GPT-4o-mini':                '#2166AC',
    'Qwen2.5-7B-Instruct':        '#1B7837',
    'Meta-Llama-3.1-8B-Instruct': '#D6604D',
    'Qalb-1.0-8B-Instruct':       '#7B2D8B',
}

print("✓ Ready")

✓ Ready


## 1. Load Dataset

In [2]:
# Main evaluation dataset — same 255 items as the baseline
df = pd.read_csv(r"P_CULTA_V2.csv")

for col in ['Topic','Power Distance','Register','Sensitivity',
            'User Role','Context','Model Role']:
    df[col] = df[col].astype(str).str.strip()
df['Sensitivity'] = df['Sensitivity'].str.capitalize()
df = df.reset_index(drop=True)
df.insert(0, 'ID', range(1, len(df)+1))

# Fixed pool of all 17 few-shot demonstrations
fewshot_df = pd.read_csv(r"fewshot_examples_17_set2.csv")
fewshot_df = fewshot_df.reset_index(drop=True)

required_demo_cols = {'User Utterance', 'Context', 'Gold Response'}
missing_demo_cols = required_demo_cols.difference(fewshot_df.columns)
assert not missing_demo_cols, f"Missing demonstration columns: {sorted(missing_demo_cols)}"
assert len(fewshot_df) == 17, f"Expected 17 demonstrations, found {len(fewshot_df)}"

print(f"Evaluation rows: {len(df)}")
print(f"Few-shot demonstrations: {len(fewshot_df)}")
print("Prompt fields: User Utterance and context")
df.head(3)

Evaluation rows: 255
Few-shot demonstrations: 17
Prompt fields: User Utterance and context


,ID,Language,Topic,User Role,Model Role,Power Distance,Register,Pragmatic Genre,Sensitivity,User Utterance,Context,Gold Response
0,1,Urdu,Refusing Offers,میزبان,مہمان,Medium,Formal,Gratitude + indirect refusal,Positive,اور سالن لے لیجیے، ابھی گرم ہے۔,آپ ایک دعوت میں مہمان ہیں۔ کھانے کے دوران میزب...,آپ کی بڑی مہربانی، واقعی بہت مزیدار تھا، مگر ا...
1,2,Urdu,Refusing Offers,دوست,دوست,Low,Informal,Polite refusal,Negative,اور چائے بنوا دوں؟,آپ اپنے دوست کے گھر ملاقات کے لیے گئے ہیں۔ گفت...,نہیں یار، تم نے پہلے ہی بہت خاطر داری کر لی ہے...
2,3,Urdu,Refusing Offers,ساس,بہو,High,Formal,Deferential refusal,Positive,بیٹا، اور روٹی لے لو۔,آپ گھر میں کھانے کی میز پر موجود ہیں۔ کھانے کے...,جی امی جان، بہت شکریہ، مگر واقعی اب مزید گنجائ...


## 2. Utterance-Only 17-Shot Set 2 Prompt Builder

In [3]:
def format_utterance_demo(ex: pd.Series) -> str:
    """Format one demonstration using context, utterance, and gold response."""
    return (
        f'Context: {ex["Context"]}\n'
        f'Utterance: "{ex["User Utterance"]}"\n'
        f'Response: "{ex["Gold Response"]}"'
    )


def build_fewshot_prompt(
    row: pd.Series,
    examples: pd.DataFrame = fewshot_df
) -> tuple[str, str]:
    """
    system : same instruction as the context + utterance baseline
    user   : 17 context + utterance → response demonstrations,
             followed by the target context + utterance

    No topic, roles, power distance, register, or sensitivity
    is shown. The target Gold Response is NEVER shown.
    """
    system = (
        "Generate a natural Urdu response. "
        "Output only the response utterance. "
        "Do not explain. "
        "Do not narrate. "
        "Do not add extra context. "
        "Do not ask unnecessary follow-up questions."
    )

    demonstrations = "\n\n---\n\n".join(
        format_utterance_demo(ex)
        for _, ex in examples.iterrows()
    )

    target = (
        f'Context: {row["Context"]}\n'
        f'Utterance: "{row["User Utterance"]}"\n'
        f'Response:'
    )

    user = (
        "Follow the response style demonstrated below and write only the "
        "response to the final Urdu utterance and context.\n\n"
        f"{demonstrations}\n\n"
        "=== FINAL CONTEXT + UTTERANCE ===\n\n"
        f"{target}"
    )

    return system, user


# Verify construction without printing all 17 demonstrations
r = df.iloc[0]
sys_msg, usr_msg = build_fewshot_prompt(r)

print(f"Demonstrations included: {len(fewshot_df)}")
print(f"SYSTEM: {sys_msg}")
print(f"Prompt characters: {len(usr_msg):,}")
print("Prompt ending:\n")
print(usr_msg[-700:])

Demonstrations included: 17
SYSTEM: Generate a natural Urdu response. Output only the response utterance. Do not explain. Do not narrate. Do not add extra context. Do not ask unnecessary follow-up questions.
Prompt characters: 5,326
Prompt ending:

شاء اللہ میں تمام انتظامات بخوبی سنبھال لوں گا۔"

---

Context: آپ نے اپنے مسافر کو آرام سے اور محفوظ طریقے سے اس کی منزل تک پہنچایا ہے۔ 
Utterance: "شکریہ جناب، امید ہے سفر آپ کے لیے آرام دہ رہا ہوگا۔"
Response: "جی بالکل، بہت شکریہ۔ یہ آپ کی اچھی خدمت کے لیے ایک چھوٹی سی ٹپ ہے، براہِ کرم قبول کیجیے۔"

---

Context: آپ اپنی اسائنمنٹ مکمل کرنے میں مصروف ہیں اور وقت کم ہونے کی وجہ سے کافی دباؤ محسوس کر رہے ہیں۔ 
Utterance: "اگر تمہیں مدد چاہیے تو میں حاضر ہوں۔"
Response: "نہیں یار، شکریہ! بس تقریباً ختم ہی ہو گیا ہے۔"

=== FINAL CONTEXT + UTTERANCE ===

Context: آپ ایک دعوت میں مہمان ہیں۔ کھانے کے دوران میزبان آپ کو مزید سالن پیش کرتا ہے۔
Utterance: "اور سالن لے لیجیے، ابھی گرم ہے۔"
Response:


## 3. Response Generation

### 3.1 GPT-4o-mini

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
from openai import OpenAI, RateLimitError

def generate_gpt(df: pd.DataFrame, api_key: str,
                 save_path: str = "responses_fs17_set2_context_utterance_gpt.csv") -> pd.DataFrame:

    client    = OpenAI(api_key=api_key)
    responses = []

    for i, row in tqdm(df.iterrows(), total=len(df), desc="GPT-4o-mini"):
        system, user = build_fewshot_prompt(row)
        retries, text = 0, "ERROR"

        while retries < 5:
            try:
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    temperature=0.3,
                    max_tokens=40,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user",   "content": user},
                    ]
                )
                text = resp.choices[0].message.content.strip()
                break
            except RateLimitError:
                wait = 2 ** retries
                print(f"  Rate limit — waiting {wait}s")
                time.sleep(wait)
                retries += 1
            except Exception as e:
                print(f"  Error row {i}: {e}")
                break

        responses.append(text)

        if i % 25 == 0 and i > 0:
            tmp = df.copy()
            tmp['GPT_Response'] = responses + [''] * (len(df) - len(responses))
            tmp.to_csv(save_path.replace('.csv','_backup.csv'),
                       index=False, encoding='utf-8-sig')
        time.sleep(1.2)

    out = df.copy()
    out['GPT_Response'] = responses
    out.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"Saved → {save_path}")
    return out
df_gpt = generate_gpt(df, OPENAI_API_KEY)
# df_gpt = pd.read_csv('responses_fs17_set2_context_utterance_gpt.csv')

### 3.2 HuggingFace Models (Qwen · LLaMA · Qalb)

In [4]:
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


def generate_hf(
    df: pd.DataFrame,
    model_id: str,
    save_path: str,
    response_col: str,
) -> pd.DataFrame:

    model_name = model_id.split("/")[-1]
    print(f"\nLoading {model_name}...")

    # -------------------------------------------------
    # Quantization
    # -------------------------------------------------
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    # -------------------------------------------------
    # Load model
    # -------------------------------------------------
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model.eval()

    # -------------------------------------------------
    # GPU Information
    # -------------------------------------------------
    print("\n================ GPU INFO ================")

    print(f"GPU : {torch.cuda.get_device_name(0)}")

    props = torch.cuda.get_device_properties(0)

    print(f"Total VRAM : {props.total_memory/1024**3:.2f} GB")

    print(
        f"Allocated : {torch.cuda.memory_allocated()/1024**3:.2f} GB"
    )

    print(
        f"Reserved  : {torch.cuda.memory_reserved()/1024**3:.2f} GB"
    )

    print("==========================================\n")

    responses = []

    # Track longest prompt encountered
    max_tokens_seen = 0

    # ======================================================
    # MAIN LOOP
    # ======================================================

    for i, row in tqdm(df.iterrows(), total=len(df), desc=model_name):

        system, user = build_fewshot_prompt(row)

        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]

        try:
            text_in = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            text_in = f"{system}\n{user}\n"

        # ======================================================
        # TOKEN DIAGNOSTICS
        # ======================================================

        num_tokens = len(tokenizer(text_in)["input_ids"])

        max_tokens_seen = max(max_tokens_seen, num_tokens)

        inputs = tokenizer(
            text_in,
            return_tensors="pt",
        ).to(model.device)

        # ======================================================
        # GENERATION
        # ======================================================

        with torch.no_grad():


            # Memory BEFORE generation
            print(
                f"Before generate : "
                f"{torch.cuda.memory_allocated()/1024**3:.2f} GB allocated | "
                f"{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved"
            )

            outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                temperature=0.3,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            
            # Memory IMMEDIATELY AFTER generation
            print(
                f"After generate  : "
                f"{torch.cuda.memory_allocated()/1024**3:.2f} GB allocated | "
                f"{torch.cuda.memory_reserved()/1024**3:.2f} GB reserved"
            )

        new_tokens = outputs[0][inputs.input_ids.shape[1]:]

        response = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        responses.append(response)

        # ======================================================
        # FREE TEMP GPU MEMORY
        # ======================================================

        del outputs
        del new_tokens
        del inputs

        gc.collect()
        torch.cuda.empty_cache()

        # ======================================================
        # PRINT DIAGNOSTICS EVERY 10 SAMPLES
        # ======================================================

        if i % 10 == 0:

            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3

            print("\n----------------------------------------")
            print(f"Sample             : {i}")
            print(f"Prompt Tokens      : {num_tokens}")
            print(f"Maximum So Far     : {max_tokens_seen}")
            print(f"Allocated VRAM     : {allocated:.2f} GB")
            print(f"Reserved VRAM      : {reserved:.2f} GB")
            print("----------------------------------------")

        # ======================================================
        # SAVE BACKUP
        # ======================================================

        if i % 25 == 0 and i > 0:

            tmp = df.copy()

            tmp[response_col] = (
                responses +
                [""] * (len(df) - len(responses))
            )

            tmp.to_csv(
                save_path.replace(".csv", "_backup.csv"),
                index=False,
                encoding="utf-8-sig",
            )

    # ======================================================
    # SAVE FINAL CSV
    # ======================================================

    out = df.copy()

    out[response_col] = responses

    out.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\nSaved -> {save_path}")

    # ======================================================
    # CLEANUP
    # ======================================================

    del model
    del tokenizer

    gc.collect()

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    print("\n============== FINAL MEMORY ==============")

    print(
        f"Allocated : {torch.cuda.memory_allocated()/1024**3:.2f} GB"
    )

    print(
        f"Reserved  : {torch.cuda.memory_reserved()/1024**3:.2f} GB"
    )

    print("==========================================")

    return out


# ======================================================
# RUN MODEL
# ======================================================


df_qalb = generate_hf(
    df,
    "enstazao/Qalb-1.0-8B-Instruct",
    "responses_fs17_set2_context_utterance_qalb.csv",
    "Qalb_Response",
)


Loading Qalb-1.0-8B-Instruct...


W0804 00:09:00.543000 6412 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


================ GPU INFO ================
GPU : NVIDIA GeForce RTX 4080 SUPER
Total VRAM : 15.99 GB
Allocated : 5.32 GB
Reserved  : 5.44 GB



Qalb-1.0-8B-Instruct:   0%|          | 0/255 [00:00<?, ?it/s]

...

Before generate : 5.32 GB allocated | 5.44 GB reserved
After generate  : 5.32 GB allocated | 9.38 GB reserved


[transformers] Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Before generate : 5.32 GB allocated | 5.44 GB reserved
After generate  : 5.32 GB allocated | 9.31 GB reserved


[transformers] Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Before generate : 5.32 GB allocated | 5.44 GB reserved
After generate  : 5.32 GB allocated | 9.42 GB reserved

Saved -> responses_fs17_set2_context_utterance_qalb.csv

============== FINAL MEMORY ==============
Allocated : 0.01 GB
Reserved  : 5.21 GB
